In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/thunthita/LidarNRBPipeline/LIDar/RawFile/")
OUT_DIR = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/Pict/")
OUT_DIR.mkdir(exist_ok=True)

DAYS = [
    "05-01-2026",
    # "06-02-2026",
]

def process_minimpl(MiniMPL: pd.DataFrame) -> pd.DataFrame:
    nrb_max = MiniMPL["copol_nrb"].max()
    if not np.isfinite(nrb_max) or nrb_max == 0:
        norm = np.nan
    else:
        norm = MiniMPL["copol_nrb"] / nrb_max

    return pd.DataFrame({
        "range_raw": MiniMPL["range_raw"],
        "range_m_for_NRB": MiniMPL["range_raw"] * 1000,
        "range_m": MiniMPL["range_nrb"] * 1000,
        "copol_raw": MiniMPL["copol_raw"],
        "copol_snr": MiniMPL["copol_snr"],
        "copol_background": MiniMPL["copol_background"],
        "copol_nrb": MiniMPL["copol_nrb"],
        "crosspol_raw": MiniMPL["crosspol_raw"],
        "crosspol_snr": MiniMPL["crosspol_snr"],
        "crosspol_background": MiniMPL["crosspol_background"],
        "crosspol_nrb": MiniMPL["crosspol_nrb"],
        "laser_energy": MiniMPL["laser_energy"],
        "pbls": MiniMPL["pbls"],
        "Normalize_copol_nrb": norm,
    })

# =========================
# Main loop (per day)
# for days where MiniMPL files end in :00 and :05 (e.g. 00:00, 00:05, ..., 23:55)
# =========================
for day in DAYS:
    day_folder = BASE_DIR / f"{day}-DATfile"
    if not day_folder.exists():
        print(f"❌ Folder missing: {day_folder}")
        continue

    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y%m%d")

    times = pd.date_range(
        start=f"{date_str} 00:00",
        end=f"{date_str} 23:55",
        freq="5min"
    )

    print(f"\n📂 Processing {day}")

    rows_day = []
    missing_count = 0
    processed_count = 0

    for t in times:
        ts_str = t.strftime("%Y%m%d%H%M")
        infile = day_folder / f"MPL_5038_{ts_str}.csv"

        if not infile.exists():
            missing_count += 1
            print(f"❌ Missing file: {infile.name}")
            continue

        try:
            MiniMPL = pd.read_csv(infile)
        except Exception as e:
            print(f"⚠️ Read error: {infile.name} → {e}")
            continue

        df = process_minimpl(MiniMPL)
        df.insert(0, "timestamp", t)
        df.insert(0, "day", day)

        rows_day.append(df)
        processed_count += 1

    if rows_day:
        MiniMPL_day = pd.concat(rows_day, ignore_index=True)

        out_file = OUT_DIR / f"MiniMPL_{day}.csv"
        MiniMPL_day.to_csv(out_file, index=False)

        print(
            f"✅ {day}: saved {processed_count} files → {out_file.name} | "
            f"❌ missing {missing_count}"
        )
    else:
        print(f"⚠️ {day}: no valid data files")



📂 Processing 05-01-2026
✅ 05-01-2026: saved 288 files → MiniMPL_05-01-2026.csv | ❌ missing 0


In [2]:
# this code is only for the day that timestamp is 00:00, 00:05 ... 23:55
from pathlib import Path
import pandas as pd
import numpy as np

IN_DIR = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/Pict/")
OUT_DIR = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/OutputPictCSV/CSVeachday/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TIMES = pd.to_timedelta(
    [f"{h:02d}:{m:02d}:00" for h in range(24) for m in (5, 35)]
)

for day in DAYS:
    in_file = IN_DIR / f"MiniMPL_{day}.csv"
    if not in_file.exists():
        print(f"❌ Missing: {in_file}")
        continue

    df = pd.read_csv(in_file)

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    df["day"] = day

    mask = df["timestamp"].dt.minute.isin([5, 35])
    df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

    df_sel = df_sel.groupby(["day", "timestamp"], as_index=False).first()

    day_ts = pd.to_datetime(day, format="%d-%m-%Y")

    full_index = pd.DataFrame({
        "day": day,
        "timestamp": day_ts + EXPECTED_TIMES
    })

    df_out = full_index.merge(df_sel, on=["day", "timestamp"], how="left")
    df_out = df_out.rename(columns={"pbls": "pbls_km"})
    df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
    df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
    df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

    df_out = df_out[
        ["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]
    ]

    out_file = OUT_DIR / f"pbls_{day}_0005_0035_to_2335.csv"
    df_out.to_csv(out_file, index=False)

    missing_count = df_out["pbls_km"].isna().sum()

    print(
        f"✅ Saved: {out_file.name} | rows={len(df_out)} "
        f"| missing={missing_count}"
    )


✅ Saved: pbls_05-01-2026_0005_0035_to_2335.csv | rows=48 | missing=0


In [3]:
MINIMPL_DIR = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/Pict")
PROTO_BASE = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/OutputPictCSV")

for day in DAYS:
    minimipl_csv = MINIMPL_DIR / f"MiniMPL_{day}.csv"
    if not minimipl_csv.exists():
        print(f"❌ Missing MiniMPL CSV: {minimipl_csv.name}")
        continue

    MiniMPL_day = pd.read_csv(minimipl_csv, parse_dates=["timestamp"])
    MiniMPL_day["timestamp"] = MiniMPL_day["timestamp"].dt.floor("min")

    print(f"\n📅 Processing {day}")



📅 Processing 05-01-2026


In [4]:
MiniMPL_day.head()


,day,timestamp,range_raw,range_m_for_NRB,range_m,copol_raw,copol_snr,copol_background,copol_nrb,crosspol_raw,crosspol_snr,crosspol_background,crosspol_nrb,laser_energy,pbls,Normalize_copol_nrb
0,05-01-2026,2026-01-05,0.029979,29.979246,119.91698,22.973406,74635.140,0.000706,0.217418,17.838509,63784.1700,0.000619,0.009371,4.301667,2.428319,0.919494
1,05-01-2026,2026-01-05,0.059958,59.958490,149.89623,9.440220,30666.166,NaN,0.235153,1.072245,3830.4692,NaN,0.011005,NaN,NaN,0.994498
2,05-01-2026,2026-01-05,0.089938,89.937740,179.87548,8.520437,27675.664,NaN,0.236075,0.681586,2434.0952,NaN,0.011251,NaN,NaN,0.998398
3,05-01-2026,2026-01-05,0.119917,119.916980,209.85472,8.234554,26750.950,NaN,0.213818,0.526134,1879.6077,NaN,0.009240,NaN,NaN,0.904269
4,05-01-2026,2026-01-05,0.149896,149.896230,239.83397,8.518830,27674.977,NaN,0.192532,0.505370,1803.9977,NaN,0.007518,NaN,NaN,0.814248
